# 14.1 提示设计 (Prompt Design)

提示设计是与大模型交互的核心技能，决定了模型输出的质量和格式。好的提示设计可以将模型能力提升数倍。

## 提示设计的核心原则

1. **明确性**：任务描述清晰无歧义
2. **具体性**：提供具体的输出格式和要求
3. **示例性**：用示例展示期望的输入输出模式
4. **结构化**：使用分隔符、标签等组织提示结构
5. **迭代性**：根据输出反馈持续优化提示

## 提示类型总览

| 类型 | 示例数 | 推理引导 | 适用场景 |
|------|--------|----------|----------|
| Zero-Shot | 0 | 无 | 简单任务 |
| Few-Shot | 1-5 | 无 | 格式化任务 |
| CoT | 1-5 | 有 | 复杂推理 |
| Zero-Shot CoT | 0 | 有 | 推理任务 |
| Multi-Prompt | 多个 | 可选 | 综合任务 |

## 1. 零样本与少样本提示

**零样本提示（Zero-Shot）**：不提供示例，直接描述任务让模型完成。适合简单、明确的任务。

**少样本提示（Few-Shot）**：提供几个示例，让模型理解任务格式和模式。适合需要特定格式的任务。

### 少样本提示的关键设计
- **示例数量**：通常3-5个，太多会占用上下文窗口
- **示例多样性**：覆盖不同情况，避免模型过度拟合某一种模式
- **示例顺序**：近期偏差（Recency Bias）——模型更关注最后的示例
- **格式一致性**：所有示例格式严格一致

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class PromptModel(nn.Module):
    def __init__(self, d=64, vocab_size=100, max_len=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 2, d * 4, batch_first=True), 1
        )
        self.head = nn.Linear(d, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)

    def generate(self, x, max_new=5, temperature=1.0):
        for _ in range(max_new):
            logits = self.forward(x)
            next_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = probs.multinomial(1)
            x = torch.cat([x, next_token], dim=1)
        return x

model = PromptModel()

zero_shot = torch.tensor([[10, 20, 30, 99]])
one_shot = torch.tensor([[10, 20, 50, 10, 21, 99]])
three_shot = torch.tensor([[10, 20, 50, 10, 21, 51, 10, 22, 52, 10, 23, 99]])

zero_out = model.generate(zero_shot, max_new=3)
one_out = model.generate(one_shot, max_new=3)
three_out = model.generate(three_shot, max_new=3)

print('=== Zero-Shot vs Few-Shot ===')
print(f'Zero-shot prompt: {zero_shot.tolist()}')
print(f'  Generated: {zero_out[0, -3:].tolist()}')
print(f'\nOne-shot prompt: {one_shot.tolist()}')
print(f'  Generated: {one_out[0, -3:].tolist()}')
print(f'\nThree-shot prompt: {three_shot.tolist()}')
print(f'  Generated: {three_out[0, -3:].tolist()}')

print(f'\nKey: More examples -> better pattern recognition.')
print(f'But more examples also use more context window and tokens.')

## 2. 思维链提示 (Chain-of-Thought)

思维链提示通过引导模型展示推理过程，显著提升复杂任务的准确率。CoT是提示工程中最重要的技术之一。

### CoT变体

**标准CoT**：在少样本示例中展示推理步骤
```
Q: 3+5=?
A: Let's add: 3+5=8. The answer is 8.
Q: 7+2=?
```

**Zero-Shot CoT**：添加"Let's think step by step"
```
Q: 7+2=?
A: Let's think step by step.
```

**Self-Consistency**：多次采样CoT路径，取多数投票

**Tree of Thought**：探索多条推理路径，回溯错误路径

In [ ]:
class ChainOfThoughtDemo:
    def __init__(self, model, vocab_size=100):
        self.model = model
        self.vocab_size = vocab_size

    def direct_prompt(self, question_tokens):
        return self.model.generate(question_tokens, max_new=1, temperature=0.1)

    def cot_prompt(self, question_tokens, reasoning_tokens):
        prompt = torch.cat([question_tokens, reasoning_tokens], dim=1)
        return self.model.generate(prompt, max_new=1, temperature=0.1)

    def self_consistency(self, question_tokens, reasoning_tokens, n_samples=10, temperature=0.7):
        answers = []
        for _ in range(n_samples):
            prompt = torch.cat([question_tokens, reasoning_tokens], dim=1)
            out = self.model.generate(prompt, max_new=1, temperature=temperature)
            answer = out[0, -1].item()
            answers.append(answer)
        from collections import Counter
        counts = Counter(answers)
        consensus = counts.most_common(1)[0]
        return answers, consensus

model = PromptModel()
cot_demo = ChainOfThoughtDemo(model)

question = torch.tensor([[5, 10, 99]])
reasoning = torch.tensor([[50, 51, 52, 53]])

direct_out = cot_demo.direct_prompt(question)
cot_out = cot_demo.cot_prompt(question, reasoning)
answers, consensus = cot_demo.self_consistency(question, reasoning, n_samples=15)

print('=== Chain-of-Thought ===')
print(f'Direct answer: {direct_out[0, -1].item()}')
print(f'CoT answer: {cot_out[0, -1].item()}')
print(f'\nSelf-Consistency ({len(answers)} samples):')
print(f'  All answers: {answers}')
from collections import Counter
print(f'  Vote counts: {dict(Counter(answers))}')
print(f'  Consensus: {consensus[0]} (confidence: {consensus[1]}/{len(answers)})')

print(f'\nKey: CoT decomposes complex reasoning into intermediate steps.')
print(f'Self-Consistency improves reliability through majority voting.')

## 3. 提示模板设计

工业级提示设计需要系统化的模板管理，确保提示的一致性和可维护性。

### 模板设计原则
- **角色设定**：明确模型的角色和能力边界
- **任务描述**：清晰描述任务目标
- **约束条件**：明确输出格式、长度、风格等约束
- **示例区**：提供few-shot示例
- **输入区**：用分隔符标记用户输入
- **输出格式**：指定输出的结构化格式

In [ ]:
class PromptTemplate:
    def __init__(self, name, system_prompt='', task_description='',
                 constraints=None, examples=None, output_format='',
                 separators=None):
        self.name = name
        self.system_prompt = system_prompt
        self.task_description = task_description
        self.constraints = constraints or []
        self.examples = examples or []
        self.output_format = output_format
        self.separators = separators or {'input': '### Input:', 'output': '### Output:'}

    def render(self, user_input=''):
        parts = []
        if self.system_prompt:
            parts.append(f'[System] {self.system_prompt}')
        if self.task_description:
            parts.append(f'[Task] {self.task_description}')
        if self.constraints:
            parts.append('[Constraints]\n' + '\n'.join(f'- {c}' for c in self.constraints))
        if self.examples:
            ex_parts = ['[Examples]']
            for i, (inp, out) in enumerate(self.examples):
                ex_parts.append(f'Example {i+1}:')
                ex_parts.append(f'  {self.separators["input"]} {inp}')
                ex_parts.append(f'  {self.separators["output"]} {out}')
            parts.append('\n'.join(ex_parts))
        if self.output_format:
            parts.append(f'[Output Format] {self.output_format}')
        if user_input:
            parts.append(f'{self.separators["input"]} {user_input}')
            parts.append(self.separators['output'])
        return '\n\n'.join(parts)

    def estimate_tokens(self, user_input=''):
        rendered = self.render(user_input)
        return len(rendered) // 4

qa_template = PromptTemplate(
    name='qa',
    system_prompt='You are a helpful and accurate assistant.',
    task_description='Answer the following question concisely and accurately.',
    constraints=['Be factual and cite sources when possible',
                 'If unsure, say so rather than guessing',
                 'Keep answers under 100 words'],
    examples=[
        ('What is the capital of France?', 'The capital of France is Paris.'),
        ('Who wrote Romeo and Juliet?', 'William Shakespeare wrote Romeo and Juliet.'),
    ],
    output_format='Direct answer, no explanation unless asked.',
)

code_template = PromptTemplate(
    name='code_generation',
    system_prompt='You are an expert Python programmer.',
    task_description='Generate Python code to solve the given problem.',
    constraints=['Use type hints', 'Include docstrings',
                 'Handle edge cases', 'Follow PEP 8'],
    examples=[
        ('Sort a list in descending order',
         'def sort_desc(lst: list) -> list:\n    return sorted(lst, reverse=True)'),
    ],
    output_format='Python code block with type hints and docstring.',
)

print('=== QA Template ===')
print(qa_template.render('What is the speed of light?'))
print(f'\nEstimated tokens: ~{qa_template.estimate_tokens("What is the speed of light?")}')

print(f'\n{"="*60}')
print(f'\n=== Code Template ===')
print(code_template.render('Find the maximum element in a list'))
print(f'\nEstimated tokens: ~{code_template.estimate_tokens("Find the maximum element in a list")}')

## 4. 提示设计最佳实践

### 提示优化技巧
1. **从简单开始**：先用zero-shot，不够再加few-shot，再不够用CoT
2. **明确格式**："以JSON格式输出"比"结构化输出"更有效
3. **角色设定**："你是专家"比"请帮我"效果更好
4. **负面示例**：有时展示"不要做什么"比"要做什么"更有效
5. **分隔符**：用```、###、---等分隔不同部分

### 常见错误
| 错误 | 示例 | 修正 |
|------|------|------|
| 指令模糊 | "写点东西" | "写一篇200字的科技评论" |
| 缺少约束 | "翻译这句话" | "翻译为正式商务英语" |
| 示例不一致 | 格式不同的示例 | 统一所有示例格式 |
| 过度复杂 | 一个提示多个任务 | 拆分为多个子任务 |
| 忽略安全 | 无输入验证 | 添加输入约束和输出审查 |

### 提示版本管理
工业实践中，提示需要像代码一样进行版本管理：
- 每个提示模板有唯一ID和版本号
- 记录提示变更和对应的性能变化
- A/B测试不同提示版本
- 回滚机制：新版本性能下降时快速回滚